# Module 02 — Lesson 2: Prompt Engineering Techniques with Claude
Anthropic's *Building with the Claude API* • Core Engineering Practices & Rigorous Evaluation

---

## 🎯 The Engineering Discipline vs. "Prompt Guessing"
In Lesson 1, we built an automated evaluation framework to measure prompt quality quantitatively. Now, we apply **systematic prompt engineering techniques** to optimize prompts against our test benchmarks.

Prompt engineering is not about lucky phrasing—it is an **iterative engineering loop**:
```
┌─────────────────┐       ┌─────────────────┐       ┌─────────────────┐
│ Candidate Prompt│ ────> │ Automated Evals │ ────> │ Failure Triage  │
└─────────────────┘       └─────────────────┘       └────────┬────────┘
         ▲                                                   │
         └───────────── Targeted Refinement ─────────────────┘
                       • Clarity & Directives
                       • Specific Constraints
                       • XML Structural Tags
                       • Few-Shot In-Context Learning
```

### The 5 Core Pillars Mastered in this Lab:
1. **Being Clear & Direct**: Eliminating vagueness with explicit operational verbs and personas.
2. **Being Specific**: Enforcing concrete constraints (word bounds, mandatory sections, numeric limits).
3. **Structuring with XML Tags**: Semantic delimiters (`<context>`, `<document>`, `<rules>`) to eliminate prompt injection and structure context.
4. **Providing Examples (Few-Shot Prompting)**: Teaching tone, schema, and edge cases (such as sarcasm) via canonical input/output pairs.
5. **The Automated Feedback Loop**: Measuring empirical improvements at each iteration with our `PromptEvaluator`.



In [ ]:
# Step 1: Environment Setup & Anthropic Client Initialization
import os
import re
import json
from textwrap import dedent
from statistics import mean
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()

# Default model configuration
CLAUDE_MODEL = os.getenv("CLAUDE_MODEL", "claude-3-5-haiku-latest")

print(f"✅ Anthropic Client Initialized. Target Model: {CLAUDE_MODEL}")



In [ ]:
# Helper Functions: Chat & Message Pipeline
def chat(messages, system=None, temperature=0.2, max_tokens=1000, stop_sequences=None):
    """Execute a Claude message call with optional system instructions and stop sequences."""
    params = {
        "model": CLAUDE_MODEL,
        "max_tokens": max_tokens,
        "messages": messages,
        "temperature": temperature,
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
        
    try:
        response = client.messages.create(**params)
        return response.content[0].text
    except Exception as e:
        # Fallback to claude-3-haiku-20240307 if the latest alias is unavailable
        params["model"] = "claude-3-haiku-20240307"
        response = client.messages.create(**params)
        return response.content[0].text

def prompt_claude(prompt_text, system_instruction=None, temperature=0.2):
    """Convenience wrapper for single-turn prompt execution."""
    return chat([{"role": "user", "content": prompt_text}], system=system_instruction, temperature=temperature)



---
## Step 2: The Benchmark Evaluation Test Suite
To measure our prompt iterations scientifically, we use a calibrated 5-case test suite based on the **Athletic Meal Planning** domain from Anthropic's curriculum.

Each test case contains:
- `prompt_inputs`: athlete dimensions, fitness goal, dietary restrictions.
- `solution_criteria`: mandatory requirements for daily calories, macronutrient grams, meal timing, and restriction compliance.



In [ ]:
# Golden Test Dataset: Athletic Meal Planner
test_dataset = [
    {
        "scenario": "Endurance Runner with Lactose Intolerance",
        "prompt_inputs": {
            "height": "178 cm",
            "weight": "68 kg",
            "goal": "Marathon training endurance with high carbohydrate needs",
            "restrictions": "Lactose intolerant (dairy-free)"
        },
        "solution_criteria": [
            "Includes daily caloric total appropriate for endurance athlete (~2800-3200 kcal)",
            "Lists macronutrient grams (protein, fat, carbohydrates)",
            "Specifies exact meal timing (e.g. 7:00 AM breakfast, 12:30 PM lunch, etc.)",
            "Contains strictly zero dairy/lactose ingredients"
        ]
    },
    {
        "scenario": "Strength Athlete with Hypercholesterolemia",
        "prompt_inputs": {
            "height": "182 cm",
            "weight": "88 kg",
            "goal": "Lean muscle hypertrophy without raising LDL cholesterol",
            "restrictions": "High cholesterol (strict low saturated fat, zero trans fat)"
        },
        "solution_criteria": [
            "Includes daily caloric total appropriate for hypertrophy (~2900-3300 kcal)",
            "Lists macronutrient grams with high lean protein (>= 160g)",
            "Specifies exact meal timing across at least 4 meals/snacks",
            "Restricts saturated fats, egg yolks, and high-cholesterol meats"
        ]
    },
    {
        "scenario": "Weight-Class Fighter Cutting Weight (Celiac Disease)",
        "prompt_inputs": {
            "height": "172 cm",
            "weight": "74 kg",
            "goal": "Cutting 4kg for upcoming competition while preserving explosive power",
            "restrictions": "Celiac disease (strict gluten-free)"
        },
        "solution_criteria": [
            "Includes moderate caloric deficit (~2000-2200 kcal)",
            "Specifies exact portion sizes in grams for all foods",
            "Contains strictly zero gluten (no wheat, barley, rye)",
            "Distributes protein evenly across pre- and post-workout windows"
        ]
    },
    {
        "scenario": "Vegetarian Swimmer with Budget Constraints",
        "prompt_inputs": {
            "height": "185 cm",
            "weight": "76 kg",
            "goal": "High daily swimming volume recovery",
            "restrictions": "Strict vegetarian (no meat/poultry/fish), low-cost ingredients"
        },
        "solution_criteria": [
            "Includes daily caloric total (~3000-3400 kcal)",
            "Provides complete plant proteins (tofu, legumes, lentils, oats)",
            "Mentions affordable pantry-friendly food choices",
            "Lists specific meal timing schedule"
        ]
    },
    {
        "scenario": "Diabetic Master CrossFitter",
        "prompt_inputs": {
            "height": "168 cm",
            "weight": "65 kg",
            "goal": "High-intensity metabolic conditioning without glucose spikes",
            "restrictions": "Type 2 Diabetes (low glycemic index, zero refined sugars)"
        },
        "solution_criteria": [
            "Specifies exact meal schedule and snack timings",
            "Lists low-GI complex carbohydrate sources and gram portions",
            "Excludes all refined sugars, juices, and high-glycemic foods",
            "Includes complete macronutrient and calorie breakdown"
        ]
    }
]

print(f"✅ Loaded {len(test_dataset)} test scenarios.")



In [ ]:
# The Evaluation Engine: Model-Based Grader
def grade_solution(test_case, solution_output):
    """Evaluate candidate prompt output against test criteria using LLM-as-a-Judge with Claude."""
    criteria_str = "\n".join([f"- {c}" for c in test_case["solution_criteria"]])
    prompt_inputs_str = json.dumps(test_case["prompt_inputs"], indent=2)

    eval_prompt = f"""
You are an expert sports nutritionist and AI evaluation judge.
Evaluate the following candidate meal plan against the scenario requirements and criteria with strict rigor.

<scenario>
{test_case["scenario"]}
</scenario>

<athlete_inputs>
{prompt_inputs_str}
</athlete_inputs>

<solution_to_evaluate>
{solution_output}
</solution_to_evaluate>

<criteria>
{criteria_str}
</criteria>

Scoring Rubric (1 to 10):
- 1-3: Critical failure (violates dietary restriction, missing calories/macros, completely uncalibrated).
- 4-6: Partial pass (respects restrictions but lacks timing, portions, or complete macronutrient metrics).
- 7-8: Good (meets all major requirements and restrictions with minor formatting/precision lapses).
- 9-10: Perfect production quality (meets 100% of criteria, clear meal timings, exact gram portions, zero restriction violations).

Output Format:
Provide your evaluation as a valid JSON object ONLY:
```json
{{
  "score": <float 1-10>,
  "passed_criteria": ["criterion 1", ...],
  "failed_criteria": ["criterion 2", ...],
  "reasoning": "<1-2 sentence concise explanation>"
}}
```
"""
    messages = [{"role": "user", "content": eval_prompt}]
    messages.append({"role": "assistant", "content": "```json"})
    raw_eval = chat(messages, stop_sequences=["```"], temperature=0.0)
    
    try:
        return json.loads(raw_eval.strip())
    except Exception:
        # Fallback regex extraction if needed
        match = re.search(r'\{[^\}]*\"score\"\s*:\s*([0-9.]+)[^\}]*\}', raw_eval, re.DOTALL)
        if match:
            return json.loads(match.group(0))
        return {"score": 5.0, "reasoning": "Parse fallback", "passed_criteria": [], "failed_criteria": []}

def run_benchmark(prompt_fn, name="Candidate"):
    """Run prompt against all test cases and compute aggregate statistics."""
    results = []
    print(f"\n🚀 Running Benchmark: [{name}]...")
    for idx, test_case in enumerate(test_dataset, 1):
        output = prompt_fn(test_case["prompt_inputs"])
        grade = grade_solution(test_case, output)
        results.append({
            "test_case": test_case["scenario"],
            "score": grade.get("score", 0.0),
            "reasoning": grade.get("reasoning", ""),
            "output_preview": output[:180].replace("\n", " ") + "..."
        })
        print(f"  • Case {idx}/5 [{test_case['scenario'][:32]}...]: Score = {grade.get('score', 0):.1f}/10")
        
    avg_score = mean([r["score"] for r in results])
    pass_rate = (len([r for r in results if r["score"] >= 7.0]) / len(results)) * 100
    print(f"📊 Summary for [{name}]: Avg Score = {avg_score:.2f}/10 | Pass Rate (>=7.0) = {pass_rate:.1f}%")
    return {"name": name, "avg_score": avg_score, "pass_rate": pass_rate, "results": results}



---
## Step 3: Baseline Iteration 0 — The Vague Prompt (Failure Mode)
Many developers start with a vague request: *"Generate a meal plan for this athlete."*

Notice what is missing:
- No role or persona.
- No requirement to calculate calories or macros.
- No requirement for timing or portions in grams.
- High risk of ignoring dietary restrictions or providing generic recipes.



In [ ]:
# Candidate 0: Vague & Unconstrained Prompt
def prompt_v0_vague(inputs):
    prompt = f"""
    Generate a meal plan for an athlete.
    Height: {inputs['height']}
    Weight: {inputs['weight']}
    Goal: {inputs['goal']}
    Restrictions: {inputs['restrictions']}
    """
    return prompt_claude(prompt)

results_v0 = run_benchmark(prompt_v0_vague, name="V0: Vague Baseline")



---
## Step 4: Iteration 1 — Being Clear & Direct (Video 2)
### The Rule:
> *"State your instructions directly using strong, imperative verbs. Eliminate polite padding, ambiguity, and vague requests. Establish a clear persona and explicit task objective."*

### Key Changes:
- Explicit directive: `You are an elite sports performance nutritionist.`
- Strong operational verb: `Generate a structured, single-day meal plan...`
- Explicit instruction to address goal and restriction directly.



In [ ]:
# Candidate 1: Clear & Direct Prompt
def prompt_v1_clear_direct(inputs):
    system_instruction = "You are an elite sports performance dietitian specializing in athletic meal programming."
    
    prompt = f"""
    Generate a structured, single-day athletic meal plan designed specifically for the following athlete profile:
    
    Athlete Profile:
    - Height: {inputs['height']}
    - Weight: {inputs['weight']}
    - Primary Goal: {inputs['goal']}
    - Dietary Restrictions: {inputs['restrictions']}
    
    Your plan must directly support the athlete's primary goal while strictly avoiding any ingredients that violate their dietary restrictions.
    """
    return prompt_claude(prompt, system_instruction=system_instruction)

results_v1 = run_benchmark(prompt_v1_clear_direct, name="V1: Clear & Direct")



---
## Step 5: Iteration 2 — Being Specific & Adding Concrete Constraints (Video 3)
### The Rule:
> *"Do not leave format, units, or timing to the model's imagination. Provide a numbered list of concrete guidelines with explicit boundaries: daily calorie targets, macronutrient grams, meal timings, and portion sizes."*

### Key Changes:
- Numbered guidelines specifying required metrics.
- Mandated meal timings (e.g. 7:00 AM, 12:30 PM).
- Mandated portion weights in grams (`g`).
- Calorie and macronutrient totals required upfront.



In [ ]:
# Candidate 2: Specific & Constrained Prompt
def prompt_v2_specific_constraints(inputs):
    system_instruction = "You are an elite sports performance dietitian specializing in athletic meal programming."
    
    prompt = f"""
    Generate a structured, single-day athletic meal plan designed specifically for the following athlete profile:
    
    Athlete Profile:
    - Height: {inputs['height']}
    - Weight: {inputs['weight']}
    - Primary Goal: {inputs['goal']}
    - Dietary Restrictions: {inputs['restrictions']}
    
    Follow these specific guidelines:
    1. State the calculated daily calorie target and macronutrient breakdown (protein, carbs, fat in grams).
    2. Divide the day into at least 4 distinct meals/snacks with exact clock times (e.g., 7:30 AM Breakfast).
    3. Specify exact portion sizes in grams (g) for every food ingredient.
    4. Adhere 100% strictly to all stated dietary restrictions; never suggest prohibited ingredients or common contaminants.
    5. Prioritize nutrient-dense, recovery-promoting whole foods tailored to the athlete's specific sport demands.
    """
    return prompt_claude(prompt, system_instruction=system_instruction)

results_v2 = run_benchmark(prompt_v2_specific_constraints, name="V2: Specific Constraints")



---
## Step 6: Iteration 3 — Structuring with XML Tags (Video 4)
### The Rule:
> *"Claude is natively trained to recognize XML tags as structural delimiters. Use tags such as `<athlete_profile>`, `<rules>`, and `<output_format>` to separate user context from instructions. This eliminates prompt injection risks and helps Claude focus attention."*

### Key Changes:
- Encapsulate inputs inside `<athlete_profile>` tags.
- Encapsulate rules inside `<guidelines>` tags.
- Provide a clean schema outline in `<output_format>`.



In [ ]:
# Candidate 3: XML-Structured Prompt
def prompt_v3_xml_structure(inputs):
    system_instruction = "You are an elite sports performance dietitian specializing in athletic meal programming."
    
    prompt = f"""
    Your task is to generate a comprehensive, single-day athletic nutrition plan.
    
    <athlete_profile>
    <height>{inputs['height']}</height>
    <weight>{inputs['weight']}</weight>
    <goal>{inputs['goal']}</goal>
    <restrictions>{inputs['restrictions']}</restrictions>
    </athlete_profile>
    
    <guidelines>
    1. Caloric Target: Calculate and state total daily calories appropriate for the athlete's body mass and training stimulus.
    2. Macronutrients: Provide total protein, carbohydrate, and fat targets in grams.
    3. Meal Schedule: Provide at least 4 distinct eating windows with specific clock timings.
    4. Portions: List every food item with exact portion weight in grams.
    5. Dietary Safety: Strict zero-tolerance for any ingredient violating the stated restrictions.
    </guidelines>
    
    <output_format>
    Follow this structure:
    # Athlete Nutritional Overview
    - Daily Target: [X] kcal | Protein: [X]g | Carbohydrates: [X]g | Fat: [X]g
    
    # Scheduled Daily Meal Plan
    - [Time] [Meal Name]: [Portion g] [Food Item] ...
    </output_format>
    """
    return prompt_claude(prompt, system_instruction=system_instruction)

results_v3 = run_benchmark(prompt_v3_xml_structure, name="V3: XML Structured")



---
## Step 7: Iteration 4 — Few-Shot In-Context Learning & Edge Cases (Video 5 & Quiz)
### The Rule:
> *"Demonstrate the expected behavior with canonical sample input/output pairs. Few-shot exemplars guide formatting precision, tone, and nuanced edge cases far more reliably than descriptive text alone."*

### Addressing Sarcasm & Complex Nuance (Quiz Question 2):
In the Coursera quiz, when Claude struggles to identify subtle sarcasm in social media posts, the proven solution is:
> **"Provide examples showing sarcastic posts labeled as negative."**

Similarly, in athletic nutrition, providing an exemplar showing exact gram weights and macro subtotals per meal teaches Claude the expected depth of detail.



In [ ]:
# Candidate 4: Few-Shot Multishot Exemplars + XML Tags
def prompt_v4_few_shot_xml(inputs):
    system_instruction = """You are an elite sports performance dietitian.
You generate precise, scientifically sound nutrition plans following exact formatting standards demonstrated in reference examples."""

    prompt = f"""
    Your task is to generate a comprehensive, single-day athletic nutrition plan.
    
    Here is an example of an input profile and its ideal output structure:
    
    <example>
    <sample_input>
    Height: 175 cm
    Weight: 72 kg
    Goal: High-volume endurance cycling
    Restrictions: Dairy-free
    </sample_input>
    <ideal_output>
    # Athlete Nutritional Overview
    - Daily Target: 2,850 kcal | Protein: 145g | Carbohydrates: 410g | Fat: 70g
    - Dietary Protocol: Strictly dairy-free (calcium fortified via plant sources)

    # Scheduled Daily Meal Plan

    ### 07:00 AM — Pre-Training Breakfast
    - 100g rolled oats cooked in 300ml unsweetened almond milk
    - 120g sliced banana
    - 30g natural peanut butter
    - 25g plant-based pea/rice protein powder
    *Meal Macros: 650 kcal | 38g Protein | 88g Carbs | 18g Fat*

    ### 12:30 PM — Post-Workout Recovery Lunch
    - 150g grilled chicken breast
    - 250g steamed jasmine rice
    - 150g steamed broccoli florets
    - 15ml extra virgin olive oil
    *Meal Macros: 740 kcal | 52g Protein | 80g Carbs | 22g Fat*

    ### 04:00 PM — Afternoon Fuel Snack
    - 60g whole wheat toast
    - 50g mashed avocado
    - 2 hard-boiled eggs
    *Meal Macros: 380 kcal | 18g Protein | 32g Carbs | 19g Fat*

    ### 07:30 PM — Nutrient Restoration Dinner
    - 180g baked Atlantic salmon
    - 220g baked sweet potato
    - 120g roasted asparagus
    *Meal Macros: 680 kcal | 46g Protein | 55g Carbs | 24g Fat*
    </ideal_output>
    </example>

    Now, generate the nutrition plan for the following athlete profile:
    
    <athlete_profile>
    <height>{inputs['height']}</height>
    <weight>{inputs['weight']}</weight>
    <goal>{inputs['goal']}</goal>
    <restrictions>{inputs['restrictions']}</restrictions>
    </athlete_profile>
    
    <guidelines>
    1. Caloric Target: Calculate and state total daily calories calibrated for the athlete.
    2. Macronutrients: Provide daily totals and per-meal breakdowns in grams.
    3. Meal Schedule: Provide at least 4 distinct eating windows with specific clock timings.
    4. Portions: List every food item with exact portion weight in grams (g).
    5. Dietary Safety: Strict zero-tolerance for any ingredient violating the stated restrictions.
    </guidelines>
    """
    return prompt_claude(prompt, system_instruction=system_instruction)

results_v4 = run_benchmark(prompt_v4_few_shot_xml, name="V4: XML + Few-Shot Exemplar")



---
## Step 8: Multi-Iteration Benchmark Comparison Table
Let's assemble the results across all 5 iterations to quantify the exact impact of each prompt engineering technique:



In [ ]:
# Benchmark Results Summary Table
all_benchmarks = [results_v0, results_v1, results_v2, results_v3, results_v4]

print("=" * 75)
print(f"{'Iteration / Prompt Strategy':<30} | {'Avg Score':<12} | {'Pass Rate (>=7.0)':<18}")
print("=" * 75)

for b in all_benchmarks:
    print(f"{b['name']:<30} | {b['avg_score']:>6.2f} / 10 | {b['pass_rate']:>12.1f}%")
print("=" * 75)

score_gain = results_v4['avg_score'] - results_v0['avg_score']
pass_gain = results_v4['pass_rate'] - results_v0['pass_rate']
print(f"\n🏆 Cumulative Engineering Gain: +{score_gain:.2f} points ({pass_gain:+.1f}% pass rate)")



---
## 🎓 Key Takeaways for Production AI Engineering
1. **Never ship vague prompts**: V0 scored poorly because language models cannot guess implicit constraints.
2. **Be clear, direct, and imperative**: Setting an explicit dietitian persona and action verb immediately stabilized responses.
3. **Specify concrete metrics**: Mandating calories, macronutrient grams, and clock times prevented subjective answers.
4. **Use XML tags to delimit context**: Tags prevent model confusion, structure multi-field inputs, and guard against accidental prompt injection.
5. **Few-shot exemplars provide formatting anchors**: A single `<example>` with `<sample_input>` and `<ideal_output>` teaches output density and style far better than lengthy paragraphs of instructions.

